## Drug Repositioning Database(RepoDB)
repoDB contains a standard set of drug repositioning successes and failures that can be used to fairly and reproducibly benchmark computational repositioning methods

### Download repoDB drug-diesases database
https://unmtid-shinyapps.net/shiny/repodb/

In [7]:
import pandas as pd

df = pd.read_csv("repoDB_drug_disease.csv", sep=",")
df.head()

,drug_name,drugbank_id,ind_name,ind_id,NCT,status,phase,DetailedStatus
0,ajmaline,DB01426,Ventricular arrhythmia,C0085612,NaN,Approved,NaN,NaN
1,ajmaline,DB01426,Supraventricular arrhythmia,C0428974,NaN,Approved,NaN,NaN
2,ajmaline,DB01426,Cardiac Arrhythmia,C0003811,NaN,Approved,NaN,NaN
3,emtricitabine,DB00879,HIV Infections,C0019693,NaN,Approved,NaN,NaN
4,enalapril,DB00584,Asymptomatic left ventricular systolic dysfunc...,C3698411,NaN,Approved,NaN,NaN


### Format the data files as the BioMedGPS format

In [8]:
# 只保留 Approved
df_approved = df[df["status"] == "Approved"]

In [26]:
formatted_df = pd.DataFrame()
formatted_df["source_name"] = df["drug_name"]
formatted_df["source_type"] = "Compound"
formatted_df["source_id"] = df["drugbank_id"]
formatted_df["target_name"] = df["ind_name"]
formatted_df["target_type"] = "Disease"
formatted_df["target_id"] = df["ind_id"]
formatted_df["relation_type"] = "RepoDB::therapeutic::Compound:Disease"
formatted_df["resource"] = "RepoDB"
formatted_df

,source_name,source_type,source_id,target_name,target_type,target_id,relation_type,resource
0,ajmaline,Compound,DB01426,Ventricular arrhythmia,Disease,C0085612,RepoDB::therapeutic::Compound:Disease,RepoDB
1,ajmaline,Compound,DB01426,Supraventricular arrhythmia,Disease,C0428974,RepoDB::therapeutic::Compound:Disease,RepoDB
2,ajmaline,Compound,DB01426,Cardiac Arrhythmia,Disease,C0003811,RepoDB::therapeutic::Compound:Disease,RepoDB
3,emtricitabine,Compound,DB00879,HIV Infections,Disease,C0019693,RepoDB::therapeutic::Compound:Disease,RepoDB
4,enalapril,Compound,DB00584,Asymptomatic left ventricular systolic dysfunc...,Disease,C3698411,RepoDB::therapeutic::Compound:Disease,RepoDB
...,...,...,...,...,...,...,...,...
13553,NaN,Compound,DB01075,"Dermatitis, Atopic",Disease,C0011615,RepoDB::therapeutic::Compound:Disease,RepoDB
13554,triamcinolone,Compound,DB00620,Vitiligo,Disease,C0042900,RepoDB::therapeutic::Compound:Disease,RepoDB
13555,oxytocin,Compound,DB00107,Obesity,Disease,C0028754,RepoDB::therapeutic::Compound:Disease,RepoDB
13556,melphalan,Compound,DB01042,Multiple Myeloma,Disease,C0026764,RepoDB::therapeutic::Compound:Disease,RepoDB


#### 对source_type and target_type进行标准化处理

In [27]:
# ---- target_id 加 UMLS: ----
t = formatted_df["target_id"].fillna("").astype(str).str.strip()
mask_t = t.ne("") & ~t.str.lower().eq("na") & ~t.str.startswith("UMLS:")
formatted_df.loc[mask_t, "target_id"] = "UMLS:" + t[mask_t]

In [28]:
# ---- source_id 加 DrugBank: ----
s = formatted_df["source_id"].fillna("").astype(str).str.strip()
mask_s = s.ne("") & ~s.str.lower().eq("na") & ~s.str.startswith("DrugBank:")
formatted_df.loc[mask_s, "source_id"] = "DrugBank:" + s[mask_s]
formatted_df

,source_name,source_type,source_id,target_name,target_type,target_id,relation_type,resource
0,ajmaline,Compound,DrugBank:DB01426,Ventricular arrhythmia,Disease,UMLS:C0085612,RepoDB::therapeutic::Compound:Disease,RepoDB
1,ajmaline,Compound,DrugBank:DB01426,Supraventricular arrhythmia,Disease,UMLS:C0428974,RepoDB::therapeutic::Compound:Disease,RepoDB
2,ajmaline,Compound,DrugBank:DB01426,Cardiac Arrhythmia,Disease,UMLS:C0003811,RepoDB::therapeutic::Compound:Disease,RepoDB
3,emtricitabine,Compound,DrugBank:DB00879,HIV Infections,Disease,UMLS:C0019693,RepoDB::therapeutic::Compound:Disease,RepoDB
4,enalapril,Compound,DrugBank:DB00584,Asymptomatic left ventricular systolic dysfunc...,Disease,UMLS:C3698411,RepoDB::therapeutic::Compound:Disease,RepoDB
...,...,...,...,...,...,...,...,...
13553,NaN,Compound,DrugBank:DB01075,"Dermatitis, Atopic",Disease,UMLS:C0011615,RepoDB::therapeutic::Compound:Disease,RepoDB
13554,triamcinolone,Compound,DrugBank:DB00620,Vitiligo,Disease,UMLS:C0042900,RepoDB::therapeutic::Compound:Disease,RepoDB
13555,oxytocin,Compound,DrugBank:DB00107,Obesity,Disease,UMLS:C0028754,RepoDB::therapeutic::Compound:Disease,RepoDB
13556,melphalan,Compound,DrugBank:DB01042,Multiple Myeloma,Disease,UMLS:C0026764,RepoDB::therapeutic::Compound:Disease,RepoDB


In [32]:
formatted_df.to_csv("formatted_RepoDB.tsv", sep="\t", index=False)

In [29]:
import os
import os.path as osp
import subprocess


def format_RepoDB(filename):
    def get_project_root():
        try:
            return osp.dirname(osp.dirname(os.getcwd()))
        except Exception as e:
            raise RuntimeError(f"Failed to determine project root: {e}")

    try:
        root_dir = get_project_root()
        print(f"Project root directory: {root_dir}")
    except RuntimeError as e:
        print(e)
        exit(1)

    database = "customdb"
    relations_path = osp.join(
        root_dir,
        "relations",
        "repoDB",
        filename,
    )
    output_dir = osp.join(
        root_dir, "formatted_relations", "repoDB"
    )
    entities_path = osp.join(root_dir, "entities.tsv")
    log_file = osp.join(output_dir, "log.txt")
    relation_types_file = osp.join(root_dir, "relation_types.tsv")

    command = [
        "graph-builder",
        "--database",
        database,
        "-d",
        relations_path,
        "-o",
        output_dir,
        "-f",
        entities_path,
        "-n",
        "20",
        "--download",
        "--skip",
        "-l",
        log_file,
        "--debug",
        "--relation-type-dict-fpath",
        relation_types_file,
        "--allow-ignore-checking-errors", 
        "all",
    ]

    print("Executing command:", " ".join(command))

    try:
        subprocess.run(command, check=True)
    except FileNotFoundError:
        print(
            "Error: 'graph-builder' command not found. Make sure it is installed and available in the PATH."
        )
        exit(1)
    except subprocess.CalledProcessError as e:
        print(f"Error: Command execution failed with return code {e.returncode}")
        print(f"Output: {e.output}")
        exit(1)
    except Exception as e:
        print(f"Unexpected error: {e}")
        exit(1)

In [34]:
format_RepoDB("formatted_RepoDB.tsv")

Project root directory: /Users/zhuzhixing/KG/biomedgps-data/graph_data
Executing command: graph-builder --database customdb -d /Users/zhuzhixing/KG/biomedgps-data/graph_data/relations/repoDB/formatted_RepoDB.tsv -o /Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/repoDB -f /Users/zhuzhixing/KG/biomedgps-data/graph_data/entities.tsv -n 20 --download --skip -l /Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/repoDB/log.txt --debug --relation-type-dict-fpath /Users/zhuzhixing/KG/biomedgps-data/graph_data/relation_types.tsv --allow-ignore-checking-errors all


2025-09-28 08:31:15 - cli:171 - INFO - Run jobs with (output_dir: /Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/repoDB, db file/directory: /Users/zhuzhixing/KG/biomedgps-data/graph_data/relations/repoDB/formatted_RepoDB.tsv, databases: ('customdb',), download: True, skip: True)
2025-09-28 08:31:18 - base_parser:229 - INFO - Using allow_ignore_checking_errors=all to ignore the checking errors.
2025-09-28 08:31:18 - customdb_parser:100 - INFO - Get 13558 relations
2025-09-28 08:31:18 - base_parser:478 - INFO - Found 13558 relations.
2025-09-28 08:31:18 - base_parser:795 - INFO - Start to get entity id map.
2025-09-28 08:31:28 - base_parser:829 - INFO - The number of deduped entity type ids: 3843
2025-09-28 08:37:43 - base_parser:839 - INFO - The number of entity ids: 3843
2025-09-28 08:37:44 - base_parser:480 - INFO - Found 3843 entity ids in entity id map.
2025-09-28 08:37:44 - base_parser:494 - INFO - The number of relations before dropna: 13558
2025-09-28 08:37:44